# importing libraries

In [1]:
import pandas as pd, re, unicodedata
from collections import Counter
from pathlib import Path

# read the data

In [2]:
df = pd.read_csv(r"E:\Courses\depi\depi-mini-and-final-projects\final-project\data\raw\raw_data.csv")
df.shape

(63209, 7)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63209 entries, 0 to 63208
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_id    63209 non-null  str  
 1   app_name     63209 non-null  str  
 2   domain       63209 non-null  str  
 3   rating       63209 non-null  int64
 4   date         63209 non-null  str  
 5   review_text  63208 non-null  str  
 6   review_type  63209 non-null  str  
dtypes: int64(1), str(6)
memory usage: 3.4 MB


# sample of the raw text

In [4]:
pd.set_option("display.max_rows", None)
sample = df["review_text"].sample(100, random_state=42)

for i, text in sample.items():
    print(i, "|", text)

48494 | جيد
18957 | خدمة ممتازه وتعامل جيد
56875 | شركة زي الزفت بتخليك تشترك ع النوتة و تبعت رسالة من نفسها و تقولك مينفعش تتلغي طب نتصل ع خدمة العملاء اتصالات تحل ال مشكلة 😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡😡
2863 | very good
5683 | ممتاز
20078 | تمام
32512 | جديد
5954 | تطبيق جميل
9670 | حظر حسابي فقط لاني دخلت من رقمين لان احد الارقام مفصول وكان يوجد لدي مال بالمحفظه عند التواصل معهم السبب اني الدخول من رقمين اساءه استخدام تطبيق غريب ولا انصح به
16033 | هو تطبيق جيد للغايه لكن في مشكله عندي طلب رحله بخصم الخصم مش بيوصل للكابتن بس بيظهر لنا كطالب فنرجو للخصم يظهر للكابتن وليه عند الطلب الرحله
33390 | جميل جداً
62804 | زفت وكل شويه الابليكشن يعلق ويقف طول اليوم مش بيفتح اسوء برنامج حقيقي زباله
17992 | حلو
44994 | My experience with Noon was extremely disappointing, and I feel the need to warn others before they go through the same situation. I purchased a refrigerator based on an advertisement that 

# normalization patterns

In [5]:
AR = "\u0621-\u064A"
EMOJI_CHARS = "\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF\u2600-\u27BF\u2B50\u23F0"

INVISIBLE = re.compile("[\u00ad\u200d\u2060\ufeff]")
LIGATURE = re.compile("[\ufdf0-\ufdff]")
COMBINING = re.compile("[\u0300-\u036f]")
LATIN_ACCENT = re.compile("[\u00c0-\u024f]")
DIACRITICS = re.compile("[\u064B-\u065F\u0670\u06D6-\u06ED]")
ALEF = re.compile("[أإآٱ]")
YA_HAMZA = re.compile(rf"(?<=[{AR}])ي+[ءئ]")
LAUGH = re.compile(rf"(?<![{AR}])ه{{2,}}(?![{AR}])")
AR_REPEAT = re.compile(rf"([{AR}])\1{{2,}}")
ALEF_GLUED = re.compile(rf"(?<![{AR}])(و?لا|و?ما)ا(?=[{AR}])(?!شا)")
ALEF_REPEAT = re.compile(rf"ا{{2,}}(?!ل[{AR}]{{3}})|(?<![{AR}])ا{{2,}}")
EN_REPEAT = re.compile(r"([a-z])\1{2,}")
MODIFIERS = re.compile("[\U0001F3FB-\U0001F3FF\u2640\u2642]")
EMOJI = re.compile(rf"[\U0001F1E6-\U0001F1FF]{{2}}|[{EMOJI_CHARS}]")
THOUSANDS = re.compile(r"(?<=[0-9])[,٬](?=[0-9]{3}(?![0-9]))")
NOT_TEXT = re.compile(rf"[^{AR} a-z0-9{EMOJI_CHARS}./:%]")
NUM_SEP = re.compile(r"(?<![0-9])[./:]|[./:](?![0-9])|(?<![0-9])%")
PERCENT_SPLIT = re.compile(rf"%(?=[{AR}a-z0-9])")
DIGIT_AR = re.compile(rf"([0-9])([{AR}])")
AR_DIGIT = re.compile(rf"([{AR}])([0-9])")
AR_LATIN = re.compile(rf"(?<=[{AR}])(?=[a-z])|(?<=[a-z])(?=[{AR}])")

JOIN_WAW = re.compile(rf"(?<![{AR}])و (?=[{AR}])(?!هه(?![{AR}]))")
JOIN_AL = re.compile(rf"(?<![{AR}])ال (?=[{AR}])")
JOIN_HA = re.compile(rf"(?<=[{AR}]) ه(?![{AR}])")
DIDI = re.compile(rf"(?<![{AR}])دي دي(?![{AR}])")
YA_REET = re.compile(rf"(?<![{AR}])يا ريت(?![{AR}])")
SAYYI = re.compile(rf"(?<![{AR}])سيي?(?![{AR}])(?! اي(?![{AR}]))")
LA_VERB = re.compile(rf"(و?)لا([يت][{AR}]{{2,}})")
GLUED_GEDAN = re.compile(rf"([{AR}]{{2,}}?)((?:جدا)+)")

APP_WORD = re.compile(r"ابل[يا]?ك[يا]?ش[يا]?ن?")
MOMTAZ = re.compile(r"م{1,2}[تن]{1,2}ا{1,2}[زذ]{1,2}(ه?)")

DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹٪٫٬", "01234567890123456789%.,")
ARABIC_VARIANTS = str.maketrans({
    "ک": "ك", "ڪ": "ك", "ګ": "ك", "گ": "ك",
    "ی": "ي", "ې": "ي", "ێ": "ي", "ۍ": "ي", "ے": "ي", "ﮯ": "ي",
    "ڤ": "ف", "پ": "ب", "چ": "ج", "ژ": "ز", "ڨ": "ق",
    "ھ": "ه", "ہ": "ه", "ە": "ه", "ۂ": "ه", "ۃ": "ه",
    "ۆ": "و", "ڵ": "ل", "ڕ": "ر", "ڛ": "س", "ڜ": "ش",
})
LATIN_VARIANTS = str.maketrans({"ı": "i", "ə": "a", "ø": "o", "ß": "s"})
APOSTROPHE = str.maketrans("", "", "'’‘`´ʼʻ")

# normalization dictionaries

In [6]:
GLUED = {
    "ياجماعه": "يا جماعه",
}

SPELLING = {
    "اورنج": "اورانج",
    "لاكن": "لكن",
    "هاذا": "هذا",
    "هاذه": "هذه",
    "ممتازا": "ممتاز",
    "ممتز": "ممتاز",
    "ممتزه": "ممتازه",
    "جدن": "جدا",
    "جدان": "جدا",
    "جددا": "جدا",
    "جدنا": "جدا",
    "جذا": "جدا",
    "جددنا": "جدا",
    "شكرن": "شكرا",
    "شكران": "شكرا",
}

WORDS = {"ف": "في", "ع": "علي"}

NOT_GLUED = (
    "لايام", "لاتمام", "لاتصال", "لايجاد", "لايقاف", "لايداع", "لايصال",
    "لاتخاذ", "لاتفاق", "لايجار", "لايمان", "لاتحاد", "لايرادات", "لايجابي",
    "لايضاح", "لايقاع", "لايهما", "لايتها", "لاتينيه",
)

PREFIXES = ("وال", "بال", "فال", "كال", "عال", "لل", "ال", "و", "ب", "ف", "ل", "ك", "")

# normalize_text

In [7]:
def collapse_phrases(words, max_n=6):
    for n in range(1, max_n + 1):
        i = 0
        while i + 2 * n <= len(words):
            if words[i:i + n] == words[i + n:i + 2 * n]:
                del words[i + n:i + 2 * n]
            else:
                i += 1
    return words


def strip_latin_accent(match):
    return unicodedata.normalize("NFD", match.group())[0]


def map_word(word):
    if word in GLUED:
        return GLUED[word]
    if word in WORDS:
        return WORDS[word]
    match = GLUED_GEDAN.fullmatch(word)
    if match:
        return f"{match.group(1)} جدا"
    match = LA_VERB.fullmatch(word)
    if match and not word.removeprefix("و").startswith(NOT_GLUED):
        return f"{match.group(1)}لا {match.group(2)}"
    for prefix in PREFIXES:
        if not word.startswith(prefix):
            continue
        stem = word[len(prefix):]
        if stem in SPELLING:
            return prefix + SPELLING[stem]
        if APP_WORD.fullmatch(stem):
            return prefix + "تطبيق"
        match = MOMTAZ.fullmatch(stem)
        if match:
            return prefix + "ممتاز" + match.group(1)
    return word

In [8]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = LIGATURE.sub(lambda m: f" {unicodedata.normalize('NFKC', m.group())} ", text)
    text = unicodedata.normalize("NFKC", text).lower()
    text = INVISIBLE.sub("", text)
    text = text.translate(DIGITS).translate(ARABIC_VARIANTS).translate(LATIN_VARIANTS).translate(APOSTROPHE)
    text = LATIN_ACCENT.sub(strip_latin_accent, text)
    text = COMBINING.sub("", DIACRITICS.sub("", text)).replace("\u0640", "")
    text = ALEF.sub("ا", text).replace("ة", "ه").replace("ى", "ي")
    text = YA_HAMZA.sub("ئ", text)

    text = LAUGH.sub("هه", text)
    text = AR_REPEAT.sub(r"\1", text)
    text = ALEF_GLUED.sub(r"\1 ا", text)
    text = ALEF_REPEAT.sub("ا", text)
    text = EN_REPEAT.sub(r"\1\1", text)

    text = MODIFIERS.sub("", text)
    text = EMOJI.sub(r" \g<0> ", text)
    text = THOUSANDS.sub("", text)
    text = NOT_TEXT.sub(" ", text)
    text = NUM_SEP.sub(" ", text)
    text = PERCENT_SPLIT.sub("% ", text)
    text = DIGIT_AR.sub(r"\1 \2", text)
    text = AR_DIGIT.sub(r"\1 \2", text)
    text = AR_LATIN.sub(" ", text)

    text = DIDI.sub("ديدي", " ".join(text.split()))
    text = YA_REET.sub("ياريت", text)
    text = " ".join(collapse_phrases(text.split()))

    text = SAYYI.sub("سئ", text)
    text = JOIN_WAW.sub("و", text)
    text = JOIN_AL.sub("ال", text)
    text = JOIN_HA.sub("ه", text)
    text = AR_REPEAT.sub(r"\1", text)

    words = " ".join(map_word(word) for word in text.split()).split()
    return " ".join(collapse_phrases(words))

# test normalize_text

In [9]:
tests = {
    "خدمة ممتازه": "خدمه ممتازه",
    "على التعرف علي المرتب": "علي التعرف علي المرتب",
    "لأن إنه": "لان انه",
    "ثوانى": "ثواني",
    "سيءه": "سئه",
    "سئ": "سئ",
    "بطيئ": "بطئ",
    "سيئة": "سئه",
    "شيء": "شئ",
    "سييء جدا": "سئ جدا",
    "يئس": "يئس",
    "قائد": "قائد",
    "جداً": "جدا",
    "جميييل جدااااا": "جميل جدا",
    "جداا": "جدا",
    "ههههه": "هه",
    "حلوههههه": "حلوه",
    "ممتاز ههههه": "ممتاز هه",
    "و ههههه": "و هه",
    "سئ سئ سئ جداً": "سئ جدا",
    "جدا و سئ سئ سئ سئ": "جدا وسئ",
    "الله أكبر الله أكبر": "الله اكبر",
    "الله": "الله",
    "ممتاز 👍": "ممتاز 👍",
    "ممتاز👍👍👍": "ممتاز 👍",
    "ممتاز 👍🏼": "ممتاز 👍",
    "😡😡😡😡": "😡",
    "😡 😡 😡": "😡",
    "❤️❤️": "❤",
    "🇪🇬🇪🇬": "🇪🇬",
    "جميل 👈 🇪🇬": "جميل 👈 🇪🇬",
    "سجود 🧙 🏻 ♀": "سجود 🧙",
    "؟؟ .. سريع": "سريع",
    "جميل و سريع": "جميل وسريع",
    "ال مشكلة": "المشكله",
    "ممتاز ه جدا": "ممتازه جدا",
    "ممتاز": "ممتاز",
    "ممتاذ": "ممتاز",
    "متاز": "ممتاز",
    "متتاز": "ممتاز",
    "منتاز": "ممتاز",
    "منتازه": "ممتازه",
    "ممتز": "ممتاز",
    "منتزه": "منتزه",
    "جميل جدان": "جميل جدا",
    "جيدجدا": "جيد جدا",
    "ممتازجدا": "ممتاز جدا",
    "شكرن": "شكرا",
    "التطبيق لايعمل": "التطبيق لا يعمل",
    "ولايعمل": "ولا يعمل",
    "لأيام طويله": "لايام طويله",
    "لاتصالات": "لاتصالات",
    "لإيجاد حل": "لايجاد حل",
    "لايف": "لايف",
    "لاانصح بالتطبيق": "لا انصح بالتطبيق",
    "ولاأحد": "ولا احد",
    "مااجي": "ما اجي",
    "ماأستخدم": "ما استخدم",
    "ماشاءالله": "ماشاءالله",
    "مااشاءالله": "ماشاءالله",
    "عشاان": "عشان",
    "خاالص": "خالص",
    "االشبكه": "الشبكه",
    "هذاالتطبيق": "هذاالتطبيق",
    "الابليكشن زفت": "التطبيق زفت",
    "الابلكيش زفت": "التطبيق زفت",
    "ابلكشين": "تطبيق",
    "اورنج احسن": "اورانج احسن",
    "لاكن": "لكن",
    "مافيش خدمه": "مافيش خدمه",
    "دي دي احسن بكتير": "ديدي احسن بكتير",
    "تطبيق سي جدا": "تطبيق سئ جدا",
    "تطبيق سي في التحميل": "تطبيق سئ في التحميل",
    "بنك سي اي بي ممتاز": "بنك سي اي بي ممتاز",
    "ف تحويل الفلوس ع الفاضي": "في تحويل الفلوس علي الفاضي",
    "بع": "بع",
    "لف": "لف",
    "بعد 5ايام": "بعد 5 ايام",
    "بس يا ريت يكون": "بس ياريت يكون",
    "10/10": "10/10",
    "5/5 ❤️": "5/5 ❤",
    "100%": "100%",
    "100%100": "100% 100",
    "نسبه 99.9%": "نسبه 99.9%",
    "١٩٫٥": "19.5",
    "٥٠٪": "50%",
    "1,500 جنيه": "1500 جنيه",
    "الساعه 10:30": "الساعه 10:30",
    "7/9/2026": "7/9/2026",
    "ريدمي9T": "ريدمي 9t",
    "a52s": "a52s",
    "للpc": "لل pc",
    "الATM": "ال atm",
    "I love shopping here Ma shaa Allah": "i love shopping here ma shaa allah",
    "doesn't work": "doesnt work",
    "I'm happy": "im happy",
    "gooood": "good",
    "güzel": "guzel",
    "très bien": "tres bien",
    "چوميا": "جوميا",
    "ڤودافون": "فودافون",
    "محمدﷺ": "محمد صلي الله عليه وسلم",
    "كاش\u200fانستا": "كاش انستا",
    "می\u200cکنم": "مي كنم",
}

results = []

for raw, expected in tests.items():
    got = normalize_text(raw)
    results.append([raw, got, expected, got == expected])

df_tests = pd.DataFrame(results, columns=["raw", "got", "expected", "ok"])
df_tests[~df_tests["ok"]]

,raw,got,expected,ok


# franco words

In [10]:
FRANCO_STRONG = {
    "mesh", "msh", "mish", "betaa3", "bta3", "bta3a", "btaa3", "keda", "kda", "kdh",
    "deh", "dah", "awi", "awy", "gedan", "gdn", "zay", "ezay", "ezzay", "leh", "lih", "lyh",
    "mafeesh", "mafish", "maafeesh", "enta", "enty", "yaani", "hena", "khalas", "khales",
    "yarab", "shokran", "tamam", "kwayes", "kwaies", "elly", "illi", "delwaqty", "ashan",
    "mashi", "ahlan",
}

FRANCO_WEAK = {"el", "al", "da", "fi", "fe", "w", "ma", "ya", "men", "mn", "bas", "ana", "ala"}

FRANCO_DIGITS = set("235679")
ENGLISH_WITH_DIGITS = {"2fa", "2nd", "3rd", "mp3", "3gp", "3ds"}

# detect_franco

In [11]:
def has_franco_digit(token):
    if token in ENGLISH_WITH_DIGITS:
        return False
    letters = sum(char.isalpha() for char in token)
    return letters >= 2 and re.search(r"(?<![0-9])[235679](?![0-9])", token) is not None


def detect_franco(text):
    if not isinstance(text, str):
        return False

    text = text.lower()
    latin = len(re.findall(r"[a-z]", text))
    arabic = len(re.findall(rf"[{AR}]", text))
    if latin == 0 or latin < arabic:
        return False

    tokens = re.findall(r"[a-z0-9]+", text)
    strong = sum(token in FRANCO_STRONG for token in tokens)
    weak = sum(token in FRANCO_WEAK for token in tokens)
    digits = sum(has_franco_digit(token) for token in tokens)
    return strong + digits >= 1 or weak >= 2

# test detect_franco

In [24]:
franco_tests = {
    "el application da mesh shaghal": True,
    "el 5edma weshha 3ala": True,
    "ana 3ayez a3ml refund": True,
    "7aga helwa": True,
    "I love shopping here Ma shaa Allah": False,
    "gooood ?": False,
    "my 4G is bad": False,
    "Samsung A52s works fine": False,
    "الشبكة ضعيفة": False,
    "التطبيق ممتاز but slow": False,
}

results = []

for raw, expected in franco_tests.items():
    got = detect_franco(raw)
    results.append([raw, got, expected, got == expected])

df_franco_tests = pd.DataFrame(results, columns=["raw", "got", "expected", "ok"])
df_franco_tests[~df_franco_tests["ok"]]

,raw,got,expected,ok


# apply on the data

In [13]:
df["normalized_text"] = df["review_text"].apply(normalize_text)
df["is_franco"] = df["review_text"].apply(detect_franco)
df.shape

(63209, 9)

# before and after

In [14]:
checks = {
    "diacritics": DIACRITICS.pattern,
    "tatweel": "\u0640",
    "alef with hamza": "[أإآٱ]",
    "ta marbuta": "ة",
    "alef maqsura": "ى",
    "ya + hamza": YA_HAMZA.pattern,
    "repeated arabic letters": AR_REPEAT.pattern,
    "repeated alef": ALEF_REPEAT.pattern,
    "laugh": rf"(?<![{AR}])ه{{3,}}(?![{AR}])",
    "symbols and punctuation": rf"[^{AR} a-z0-9{EMOJI_CHARS}./:%]|(?<![0-9])[./:%]|[./:](?![0-9])",
    "arabic digits": "[٠-٩۰-۹]",
    "thousands comma": "[0-9],[0-9]",
    "invisible chars": "[\u00ad\u200b-\u200f\u202a-\u202e\u2060-\u2064\u2066-\u206f\ufeff]",
    "latin accents": "[\u00c0-\u024f\u0300-\u036f]",
    "persian and urdu letters": "[کیڤپچگژھہەۃے]",
    "glued arabic and latin": rf"[{AR}][a-z]|[a-z][{AR}]",
    "glued digit and arabic": rf"[0-9٠-٩][{AR}]|[{AR}][0-9٠-٩]",
    "glued waw": JOIN_WAW.pattern,
    "glued al": JOIN_AL.pattern,
    "space before ha": JOIN_HA.pattern,
    "sayyi": SAYYI.pattern,
    "ya reet": YA_REET.pattern,
    "didi": DIDI.pattern,
    "fa and ain alone": rf"(?<![{AR}a-z0-9])[فع](?![{AR}a-z0-9])",
    "double spaces": "  ",
}


def count_rows(column, pattern):
    return column.fillna("").apply(lambda text: bool(re.search(pattern, text.lower()))).sum()


def count_glued_la(column):
    def has_glued(text):
        return any(
            LA_VERB.fullmatch(word) and not word.removeprefix("و").startswith(NOT_GLUED)
            for word in text.split()
        )
    return column.fillna("").apply(has_glued).sum()


report = pd.DataFrame({
    "before": {name: count_rows(df["review_text"], p) for name, p in checks.items()},
    "after": {name: count_rows(df["normalized_text"], p) for name, p in checks.items()},
})
report.loc["glued laa + verb"] = [
    count_glued_la(df["review_text"].str.replace("[أإآ]", "ا", regex=True)),
    count_glued_la(df["normalized_text"]),
]

report

,before,after
diacritics,2064,0
tatweel,68,0
alef with hamza,8244,0
ta marbuta,12578,0
alef maqsura,7329,0
ya + hamza,3795,0
repeated arabic letters,1358,0
repeated alef,1177,0
laugh,16,0
symbols and punctuation,7351,0


# review changed rows

In [15]:
pd.set_option("display.max_colwidth", 80)
changed = df[df["review_text"].fillna("") != df["normalized_text"]]
changed[["review_text", "normalized_text"]].sample(25, random_state=7)

,review_text,normalized_text
42492,ممتاز ❤️❤️❤️ ارجو تطوير ليكون اسرع +بصمه صوتيه + بصمه اليد + الرقم السرى,ممتاز ❤ ارجو تطوير ليكون اسرع بصمه صوتيه بصمه اليد الرقم السري
42888,البرنامج مش بيفتح خالص لا علي بيانات الهاتف ولا علي الواي فاي إيه المشكله مش...,البرنامج مش بيفتح خالص لا علي بيانات الهاتف ولا علي الواي فاي ايه المشكله مش...
50699,الابلكيشن تقيل جدا وهوه بيفتح,التطبيق تقيل جدا وهوه بيفتح
46082,خدمة رائعة,خدمه رائعه
58149,ارجو العمل علي تحسين البرنامج وتحديثة بحيث يكون اكثر سهوله التعامل مع خدمة ا...,ارجو العمل علي تحسين البرنامج وتحديثه بحيث يكون اكثر سهوله التعامل مع خدمه ا...
7948,افضل وارقى المعاملاااات,افضل وارقي المعاملات
60226,شركة زبالة انا نفسي أعرف هم مش عايزين يعملوا انترنت غير محدود ليه . علشان هم...,شركه زباله انا نفسي اعرف هم مش عايزين يعملوا انترنت غير محدود ليه علشان هما ...
10127,سئ جدا كل لما كابتن يقبل المشوار واستناه وقت يرجع ويلغى الطلب بعد وقت بصراحة...,سئ جدا كل لما كابتن يقبل المشوار واستناه وقت يرجع ويلغي الطلب بعد وقت بصراحه...
58433,التطبيق حاليا مش شغال في فودافون كاش بعد التحديث الواجهة اتغيرت لكن مش قابل ...,التطبيق حاليا مش شغال في فودافون كاش بعد التحديث الواجهه اتغيرت لكن مش قابل ...
61660,تطبيق بينزل تحديث كل ثانية,تطبيق بينزل تحديث كل ثانيه


# review franco rows

In [16]:
print(df["is_franco"].mean())
df[df["is_franco"]][["review_text", "normalized_text"]]

0.00015820531886282017


,review_text,normalized_text
1357,"Amazon global bir uygulama olduğundan, kullanıcıların özgürce seçim yapabilm...",amazon global bir uygulama oldugundan kullanicilarin ozgurce secim yapabilme...
14708,Gamed gedan,gamed gedan
28555,taxs al3sama,taxs al3sama
29718,a7sn tatbi9,a7sn tatbi9
30184,sara7a ttbi9 dyl indreve chi 7ja li nf3atna o khososa talaba,sara7a ttbi9 dyl indreve chi 7ja li nf3atna o khososa talaba
36540,m5damt liya knbri nsri mki9blot liya,m5damt liya knbri nsri mki9blot liya
38278,9and n njY,9and n njy
38611,Ana kanbrit nakhod man3ankom walking makaynachi la livraison lnaqahi,ana kanbrit nakhod man3ankom walking makaynachi la livraison lnaqahi
56015,VF və digər sahələrdə olduğu halda bu t və k bir hissəsi isə bu rəqəm daha d...,vf va digar sahalarda oldugu halda bu t va k bir hissasi isa bu raqam daha d...
60198,Ibrahim Abu Al-Majd Al-Sayyid Hassan,ibrahim abu al majd al sayyid hassan


# rare characters

In [17]:
OK_CHAR = re.compile(rf"[{AR} a-z0-9{EMOJI_CHARS}./:%]")
rare = Counter(char for text in df["normalized_text"] for char in text if not OK_CHAR.match(char))
rare.most_common(10)

[]

# empty rows

In [18]:
empty = df[df["normalized_text"] == ""]
empty[["review_text", "normalized_text"]]

,review_text,normalized_text
10073,(^ー^),
14900,Супер,
27029,NaN,
36366,. .,
52753,很好！,


# top words

In [19]:
words = Counter(word for text in df["normalized_text"] for word in text.split())
pd.DataFrame(words.most_common(300), columns=["word", "count"]).head(100)

,word,count
0,جدا,13365
1,ممتاز,10721
2,في,8884
3,من,8111
4,التطبيق,8006
5,تطبيق,6839
6,مش,5717
7,لا,5486
8,علي,4801
9,سئ,4551


# sample of the normalized text

In [20]:
pd.set_option("display.max_rows", None)
sample = df["normalized_text"].sample(100, random_state=30)

for i, text in sample.items():
    print(i, "|", text)

59007 | جميله
36227 | اسوا تجربه واسوا شركه شحن بيتكلم معاك كانه انسان الي وبيكنسلو الاوردرات اللي انت دافع تمنها اون لاين ويرجعولك فلوسك بعد 14 يوم وطبعا الحاجه سعرها بيرفع تاني في مش بتعرف تشتريها عن تجربه ملحوظه الاوردر اتكنسل من غير ما يرجعولي اصلا اللي هو اخبط دماغك في الحيط وجيت اطلبه تاني لقيت فرق 50 في المئه
34369 | ممتاز جدا اشكركم
48645 | ممتاز
30218 | اكثر السيارات كانك راكب كنبه ومش نظيفه حتي السوقين راحتهم مش نظيفه وعرق وسجاير والاجره مبالغ فيها جدا
53260 | ماشاءالله تطبيق ممتاز جدا والله ممتاز
60414 | سئ جدا
48984 | ممتاز
41871 | تطبيق فيه كل اللي عايزه تطبيق روعه ناقصه حاجه مهمه للجميع بعد اي انجاز تحويل او استلام تحويل ان استطيع اعمل اسكرين شوت او احمل ملف او صوره بالعمليه المحدده ياريت دا يكون تطوير زي الانستا لما بعمل عمليه بعمل مشاركه الايصال للعميل فبيطمئن ان فلوسه وصلت
62717 | زباله
15313 | ممتازه
54261 | ممتاز
19304 | جميل
30093 | هذا البرنامج ممتاز فهو يمكنك من التنقل بسرعه وسهوله بمبلغ صغير يستطيع الجميع توفيره
504 | تجربه سئه جدا وصلني الطلب تالف وعملت طلب استب

In [21]:
df.head(10)

,review_id,app_name,domain,rating,date,review_text,review_type,normalized_text,is_franco
0,2e6046b5-4407-45ad-8487-c1ef349bae02,Amazon,E-commerce,5,2026-09-29,l love shopping here Ma shaa Allah,Positive,l love shopping here ma shaa allah,False
1,fb178bf4-ef23-4227-980d-284a8905a57a,Amazon,E-commerce,5,2026-09-29,الله واكبر لاله الا الله والله واكبر سبحان الله الخالق وحده حي لا يموت لا يم...,Positive,الله واكبر لاله الا الله والله واكبر سبحان الله الخالق وحده حي لا يموت رب ال...,False
2,2856ab4b-4270-486b-ab56-aeb9116cf879,Amazon,E-commerce,1,2026-09-29,التطبيق فية لاج الصور كلها سودة,Negative,التطبيق فيه لاج الصور كلها سوده,False
3,cc842fc5-c895-4246-af12-df33a1e8f087,Amazon,E-commerce,5,2026-09-29,gooood ?,Positive,good,False
4,c7f05214-0917-4e24-b6f1-1d97d9effef7,Amazon,E-commerce,4,2026-09-29,"كپعا دك.ك،،،،،،،،,,،,،،،،،،،،،،،،،،,,74%,چ,ؤليءر،ى كد! ف دن د في،ففءفلف،دد. ...",Positive,كبعا دك ك 74% ج ؤلئر ي كد في دن د في ففءفلف دد ظ ظكدكدزن طحكجحطحظظزمزظ ربي ع...,False
5,a07e1c62-0b3c-451e-9631-f8804351e09f,Amazon,E-commerce,5,2026-09-28,الثقه السرعه في التوصيل الامانه أمازون رقم واحد في مصر,Positive,الثقه السرعه في التوصيل الامانه امازون رقم واحد في مصر,False
6,a598f690-b72d-4a6f-b3f7-ea6dadd052c5,Amazon,E-commerce,1,2026-09-28,تجربة سيئة جدا,Negative,تجربه سئه جدا,False
7,72d3a0e9-1d59-408d-8ec6-13c888de62d2,Amazon,E-commerce,3,2026-09-28,يرجا اضافة الغلة العربية في تركيا,Neutral,يرجا اضافه الغله العربيه في تركيا,False
8,4d24fb5f-e74b-430c-82d3-5297be9915a5,Amazon,E-commerce,1,2026-09-28,تطبيق سيئ اردت إرجاع منتج اشتريته من حسابي الثاني ولديهم عنواني ورقم موبايلي...,Negative,تطبيق سئ اردت ارجاع منتج اشتريته من حسابي الثاني ولديهم عنواني ورقم موبايلي ...,False
9,10c963fd-60be-46bb-a531-6a00071fef01,Amazon,E-commerce,3,2026-09-28,جميل,Neutral,جميل,False


# save

In [ ]:
#out = Path(r"E:\Courses\depi\depi-mini-and-final-projects\final-project\data\cleaned\cleaned_a.csv")
#out.parent.mkdir(parents=True, exist_ok=True)
#df.to_csv(out, index=False, encoding="utf-8-sig")